In [1]:
# %%
from pathlib import Path
import re
import hashlib

import pandas as pd
import duckdb
from tqdm import tqdm

from transformers import AutoTokenizer

c:\Users\aarav\anaconda3\envs\pmc_graphrag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# %%
PROJECT_ROOT = Path("..").resolve()

PARSED_DIR = PROJECT_ROOT / "data" / "parsed"
RAW_PARQUET = PARSED_DIR / "pmc_raw_articles.parquet"
OUT_PARQUET = PARSED_DIR / "pmc_retrieval_candidates.parquet"

assert RAW_PARQUET.exists(), "Input parquet from Notebook 01 not found"

# Chunking params (LOCKED)
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_TOKENS = 350
CHUNK_OVERLAP = 50

In [3]:
# %%
df = pd.read_parquet(RAW_PARQUET)

print("Rows:", len(df))
print("Unique PMCIDs:", df["pmcid"].nunique())
df.head(3)

Rows: 88995
Unique PMCIDs: 994


,pmcid,version,article_title,section_raw,paragraph_index,paragraph_text,license,is_retracted
0,PMC12446057,1,The immune checkpoint TIM-3/HMGB-1 axis in myo...,Introduction,0,Immune checkpoints (ICs) serve as switches on ...,CC BY,False
1,PMC12446057,1,The immune checkpoint TIM-3/HMGB-1 axis in myo...,Introduction,1,The two most studied ICs are cytotoxic T-lymph...,CC BY,False
2,PMC12446057,1,The immune checkpoint TIM-3/HMGB-1 axis in myo...,Introduction,2,"Noteworthy is that some TIM-3 ligands, such as...",CC BY,False


In [4]:
# %%
def normalize_section(section: str) -> str:
    s = section.lower()

    if "result" in s:
        return "results"
    if "discussion" in s:
        return "discussion"

    return "other"


df["section_norm"] = df["section_raw"].apply(normalize_section)

print(df["section_norm"].value_counts())

section_norm
other         68157
results       13232
discussion     7606
Name: count, dtype: int64


In [5]:
# %%
df_filt = df[df["section_norm"].isin(["results", "discussion"])].copy()

print("Filtered rows:", len(df_filt))
print("Filtered PMCIDs:", df_filt["pmcid"].nunique())

Filtered rows: 20838
Filtered PMCIDs: 814


In [6]:
# %%
df_filt = df_filt.sort_values(
    ["pmcid", "section_norm", "paragraph_index"]
)

grouped = (
    df_filt
    .groupby(["pmcid", "article_title", "section_norm", "license"], as_index=False)
    .agg({"paragraph_text": lambda x: "\n".join(x)})
)

grouped.rename(
    columns={"paragraph_text": "section_text"},
    inplace=True
)

print("Aggregated sections:", len(grouped))
grouped.head(2)

Aggregated sections: 1464


,pmcid,article_title,section_norm,license,section_text
0,PMC12446057,The immune checkpoint TIM-3/HMGB-1 axis in myo...,discussion,CC BY,"We investigated, for the first time, expressio..."
1,PMC12446057,The immune checkpoint TIM-3/HMGB-1 axis in myo...,results,CC BY,Baseline characteristics of the Glycometabolic...


In [7]:
# %%
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [8]:
# %%
def chunk_text(text: str, tokenizer, max_tokens=350, overlap=50):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []

    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens)

        chunks.append((chunk_text, len(chunk_tokens)))
        start += max_tokens - overlap

    return chunks

In [9]:
# %%
rows = []

for _, row in tqdm(grouped.iterrows(), total=len(grouped)):
    chunks = chunk_text(
        row["section_text"],
        tokenizer,
        CHUNK_TOKENS,
        CHUNK_OVERLAP,
    )

    for idx, (txt, tok_ct) in enumerate(chunks):
        rows.append({
            "pmcid": row["pmcid"],
            "article_title": row["article_title"],
            "section": row["section_norm"],
            "chunk_index": idx,
            "chunk_text": txt.strip(),
            "token_count": tok_ct,
            "license": row["license"],
        })

chunks_df = pd.DataFrame(rows)
print("Total chunks:", len(chunks_df))
chunks_df.head(3)

100%|██████████| 1464/1464 [00:09<00:00, 147.73it/s]

Total chunks: 13162


,pmcid,article_title,section,chunk_index,chunk_text,token_count,license
0,PMC12446057,The immune checkpoint TIM-3/HMGB-1 axis in myo...,discussion,0,"we investigated, for the first time, expressio...",350,CC BY
1,PMC12446057,The immune checkpoint TIM-3/HMGB-1 axis in myo...,discussion,1,"an mi. on pbmc level, overall tim - 3 rna expr...",350,CC BY
2,PMC12446057,The immune checkpoint TIM-3/HMGB-1 axis in myo...,discussion,2,"to be more organ - bound, possibly to maintain...",350,CC BY


In [10]:
# %%
def norm_text(t: str) -> str:
    t = t.lower()
    t = re.sub(r"\s+", " ", t)
    return t.strip()


chunks_df["text_norm"] = chunks_df["chunk_text"].apply(norm_text)
chunks_df["text_hash"] = chunks_df["text_norm"].apply(
    lambda x: hashlib.sha1(x.encode("utf-8")).hexdigest()
)

before = len(chunks_df)
chunks_df = chunks_df.drop_duplicates("text_hash")
after = len(chunks_df)

print(f"Deduplicated: {before} → {after}")

Deduplicated: 13162 → 13160


In [11]:
# %%
def make_chunk_id(row):
    base = f"{row.pmcid}|{row.section}|{row.chunk_index}|{row.text_hash}"
    return hashlib.sha1(base.encode("utf-8")).hexdigest()


chunks_df["chunk_id"] = chunks_df.apply(make_chunk_id, axis=1)

In [12]:
# %%
MIN_TOKENS = 50

before = len(chunks_df)
chunks_df = chunks_df[chunks_df["token_count"] >= MIN_TOKENS].reset_index(drop=True)
after = len(chunks_df)

print(f"Filtered short chunks: {before} → {after}")

Filtered short chunks: 13160 → 12935


In [13]:
# %%
final_cols = [
    "chunk_id",
    "pmcid",
    "article_title",
    "section",
    "chunk_index",
    "chunk_text",
    "token_count",
    "license",
]

final_df = chunks_df[final_cols].reset_index(drop=True)

print("Final rows:", len(final_df))
final_df.head(3)

Final rows: 12935


,chunk_id,pmcid,article_title,section,chunk_index,chunk_text,token_count,license
0,9fd0d269d90d3d89b02c81308d3a1500b6b91c37,PMC12446057,The immune checkpoint TIM-3/HMGB-1 axis in myo...,discussion,0,"we investigated, for the first time, expressio...",350,CC BY
1,79b3541338632b7d1dd88f1b98e362958d8a7411,PMC12446057,The immune checkpoint TIM-3/HMGB-1 axis in myo...,discussion,1,"an mi. on pbmc level, overall tim - 3 rna expr...",350,CC BY
2,5020b51f114f3fc6edf8746b2206e74869294b06,PMC12446057,The immune checkpoint TIM-3/HMGB-1 axis in myo...,discussion,2,"to be more organ - bound, possibly to maintain...",350,CC BY


In [14]:
# %%
final_df.to_parquet(OUT_PARQUET, index=False)

print("Saved retrieval candidates to:")
print(OUT_PARQUET.resolve())

Saved retrieval candidates to:
D:\Pictures\pmc_graphrag\data\parsed\pmc_retrieval_candidates.parquet


In [15]:
# %%
print("Chunks per article (summary):")
display(final_df.groupby("pmcid").size().describe())

print("\nSections:")
print(final_df["section"].value_counts())

print("\nToken count stats:")
print(final_df["token_count"].describe())

Chunks per article (summary):


count    814.000000
mean      15.890663
std       10.726802
min        1.000000
25%        9.000000
50%       14.000000
75%       21.000000
max       73.000000
dtype: float64


Sections:
section
results       7782
discussion    5153
Name: count, dtype: int64

Token count stats:
count    12935.000000
mean       332.817781
std         56.150023
min         50.000000
25%        350.000000
50%        350.000000
75%        350.000000
max        350.000000
Name: token_count, dtype: float64


In [16]:
# =========================
# NOTEBOOK 02 — EVIDENCE LAYER METRICS REPORT
# =========================
import pandas as pd
import numpy as np

print("=" * 90)
print("NOTEBOOK 02 METRICS — SECTION FILTERING + CHUNKING + DEDUPE")
print("=" * 90)

if 'df' in globals():
    print(f"Raw rows loaded: {len(df):,}")
    print(f"Raw unique PMCIDs: {df['pmcid'].nunique():,}")

if 'df_filt' in globals():
    print(f"\nFiltered paragraph rows (Results/Discussion): {len(df_filt):,}")
    print(f"Filtered unique PMCIDs: {df_filt['pmcid'].nunique():,}")

    print("\nFiltered section distribution:")
    display(df_filt["section_norm"].value_counts(dropna=False).to_frame("count"))

if 'grouped' in globals():
    print(f"\nAggregated sections: {len(grouped):,}")
    grouped_len = grouped["section_text"].str.len()
    print("\nSection text length stats:")
    display(grouped_len.describe())

if 'chunks_df' in globals():
    print(f"\nChunk rows before final column trim: {len(chunks_df):,}")
    if "token_count" in chunks_df.columns:
        print("\nChunk token stats:")
        display(chunks_df["token_count"].describe())

    if "section" in chunks_df.columns:
        print("\nChunk section distribution:")
        display(chunks_df["section"].value_counts(dropna=False).to_frame("count"))

# Dedupe diagnostics
if 'chunks_df' in globals() and "text_hash" in chunks_df.columns:
    total_rows = len(chunks_df)
    unique_hashes = chunks_df["text_hash"].nunique()
    duplicate_rows = total_rows - unique_hashes
    dedupe_rate = duplicate_rows / total_rows if total_rows else np.nan

    print("\nDedupe diagnostics:")
    print(f"Total chunk rows considered: {total_rows:,}")
    print(f"Unique normalized text hashes: {unique_hashes:,}")
    print(f"Duplicate chunk rows removed or removable: {duplicate_rows:,}")
    print(f"Duplicate rate: {dedupe_rate:.2%}" if pd.notna(dedupe_rate) else "Duplicate rate: N/A")

    dup_hashes = chunks_df["text_hash"].value_counts()
    cross_article_dup = (
        chunks_df.groupby("text_hash")["pmcid"].nunique()
        .rename("pmcid_nunique")
        .reset_index()
    )
    cross_article_dup = cross_article_dup[cross_article_dup["pmcid_nunique"] > 1]

    print(f"Duplicate text hashes appearing across multiple PMCIDs: {len(cross_article_dup):,}")

# Final retrieval candidate diagnostics
if 'final_df' in globals() and not final_df.empty:
    print(f"\nFinal retrieval candidates: {len(final_df):,}")
    print(f"Final unique PMCIDs: {final_df['pmcid'].nunique():,}")
    print(f"Final unique chunk_ids: {final_df['chunk_id'].nunique():,}")

    print("\nChunks per article:")
    display(final_df.groupby("pmcid").size().describe())

    print("\nChunks per section:")
    display(final_df["section"].value_counts(dropna=False).to_frame("count"))

    print("\nToken count stats (final):")
    display(final_df["token_count"].describe())

    print("\nShort-chunk filter diagnostics:")
    if 'MIN_TOKENS' in globals():
        below_min = (final_df["token_count"] < MIN_TOKENS).sum()
        print(f"MIN_TOKENS threshold: {MIN_TOKENS}")
        print(f"Rows below threshold remaining: {below_min:,}")

    print("\nArticle coverage after chunking:")
    article_chunk_counts = final_df.groupby("pmcid").size()
    print(f"Articles with at least 1 retrieval chunk: {article_chunk_counts.shape[0]:,}")
    print(f"Median chunks/article: {article_chunk_counts.median():.1f}")
    print(f"Mean chunks/article: {article_chunk_counts.mean():.2f}")

print("\nOutput parquet:")
print(OUT_PARQUET.resolve() if 'OUT_PARQUET' in globals() else "N/A")

NOTEBOOK 02 METRICS — SECTION FILTERING + CHUNKING + DEDUPE
Raw rows loaded: 88,995
Raw unique PMCIDs: 994

Filtered paragraph rows (Results/Discussion): 20,838
Filtered unique PMCIDs: 814

Filtered section distribution:


,count
section_norm,
results,13232
discussion,7606



Aggregated sections: 1,464

Section text length stats:


count     1464.000000
mean     10765.078552
std       8215.404892
min        237.000000
25%       5849.250000
50%       8651.000000
75%      12757.750000
max      87176.000000
Name: section_text, dtype: float64


Chunk rows before final column trim: 12,935

Chunk token stats:


...


Chunks per section:


,count
section,
results,7782
discussion,5153



Token count stats (final):


count    12935.000000
mean       332.817781
std         56.150023
min         50.000000
25%        350.000000
50%        350.000000
75%        350.000000
max        350.000000
Name: token_count, dtype: float64


Short-chunk filter diagnostics:
MIN_TOKENS threshold: 50
Rows below threshold remaining: 0

Article coverage after chunking:
Articles with at least 1 retrieval chunk: 814
Median chunks/article: 14.0
Mean chunks/article: 15.89

Output parquet:
D:\Pictures\pmc_graphrag\data\parsed\pmc_retrieval_candidates.parquet
